# Split Dataset

**Split Ratio:** 70% Train / 15% Validation / 15% Test  

## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


## 2. Konfigurasi


In [ ]:
# Path dataset (Google Drive)
glioma           = '/content/drive/MyDrive/Tugas Akhir/dataset/glioma'
meningioma       = '/content/drive/MyDrive/Tugas Akhir/dataset/meningioma'
image_tumor_mask = '/content/drive/MyDrive/Tugas Akhir/dataset/image_tumor_mask'
mask_tumor       = '/content/drive/MyDrive/Tugas Akhir/dataset/mask_tumor'
no_tumor         = '/content/drive/MyDrive/Tugas Akhir/dataset/no_tumor'

# Path output
OUTPUT_ROOT = '/content/drive/MyDrive/Tugas Akhir/dataset_split_val_test_FINAL'

# Rasio split
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Mode image_tumor_mask (1 sumber, rawan leakage jika "free"):
#  "test_only" -> semua ke test (disarankan untuk XAI)
#  "val_test"  -> dibagi 50/50 val & test
#  "free"      -> ikut split random 70/15/15
IMAGE_TUMOR_MASK_MODE = "val_test"

# Opsi lain
RANDOM_SEED   = 42      # reproduktibilitas
COPY_FILES    = True    # False = dry-run (hanya statistik, tanpa copy)
SAVE_MANIFEST = True

OUTPUT_EXT        = '.png'  # semua gambar dikonversi ke PNG agar seragam
IMAGE_EXTENSIONS  = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}


In [ ]:
import os
from pathlib import Path
from PIL import Image as _PILImage

def rename_files(folder, prefix, extensions=None):
    """Rename file gambar di folder jadi prefix_1<ext>, prefix_2<ext>, dst.
    Ekstensi asli dipertahankan; konversi ke PNG dilakukan saat copy ke output."""
    if extensions is None:
        extensions = IMAGE_EXTENSIONS

    folder = Path(folder)
    if not folder.exists():
        print(f"Folder tidak ditemukan: {folder}")
        return

    files = sorted([f for f in folder.iterdir() if f.is_file() and f.suffix.lower() in extensions])
    if not files:
        print(f"Tidak ada file gambar di: {folder}")
        return

    # Rename ke nama sementara dulu agar tidak konflik antar file
    temp_names = []
    for f in files:
        tmp = f.with_name(f"__tmp_{f.name}")
        f.rename(tmp)
        temp_names.append(tmp)

    # Pakai ekstensi asli (tmp.suffix), bukan hardcode .png
    for idx, tmp in enumerate(temp_names, start=1):
        tmp.rename(folder / f"{prefix}_{idx}{tmp.suffix}")

rename_files(image_tumor_mask, prefix='tumor_mask')
rename_files(mask_tumor,       prefix='mask')


## 3. Import Library & Fungsi Utilitas


In [ ]:
import os
import csv
import json
import shutil
import random
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from sklearn.model_selection import StratifiedGroupKFold
from PIL import Image as PILImage

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

def get_images(folder):
    folder = Path(folder)
    if not folder.exists():
        print(f"Folder tidak ditemukan: {folder}")
        return []
    return sorted([f for f in folder.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS])

def get_patient_id(filepath):
    # Format nama "{patientID}_{frameNum}" -> ambil sebelum underscore pertama.
    # Jika bukan angka (mis. "no_tumor_3"), pakai stem penuh agar tiap file dianggap pasien unik.
    stem = Path(filepath).stem
    part = stem.split('_')[0]
    return part if part.isdigit() else stem

def copy_file_safe(src, dest_dir):
    # Nama output "{folder}__{stem}.png" agar file dari folder berbeda (glioma/meningioma)
    # dengan stem sama tidak saling menimpa.
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    src  = Path(src)
    dest = dest_dir / (src.parent.name + "__" + src.stem + OUTPUT_EXT)
    if not dest.exists():
        if src.suffix.lower() == OUTPUT_EXT:
            shutil.copy2(src, dest)
        else:
            img = PILImage.open(src).convert('RGB')
            img.save(dest, format='PNG', optimize=False)

def bar(n, total, width=20):
    filled = int(width * n / total) if total > 0 else 0
    return '█' * filled + '░' * (width - filled)

def split_stratified_group(files, labels, groups, train_r, val_r, random_state=42):
    """Split 2 tahap (train | val+test -> val | test) via StratifiedGroupKFold.
    Menjamin: tidak ada grup (pasien) bocor antar split & proporsi kelas terjaga."""
    X = np.arange(len(files))
    y = np.array(labels)
    g = np.array(groups)

    n_unique_groups = len(set(groups))
    n_unique_labels = len(set(labels))

    # Tahap 1: train vs sisa
    n_splits_1 = min(max(2, round(1 / (1 - train_r))), n_unique_groups, n_unique_labels)
    if n_splits_1 < 2:
        raise ValueError(f"Grup/kelas tidak cukup untuk split tahap 1 (grup={n_unique_groups}, kelas={n_unique_labels})")

    sgkf1 = StratifiedGroupKFold(n_splits=n_splits_1, shuffle=True, random_state=random_state)
    train_idx, rest_idx = next(iter(sgkf1.split(X, y, g)))

    # Tahap 2: val vs test dari sisa
    val_frac_of_rest = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    g_rest, y_rest = g[rest_idx], y[rest_idx]
    n_unique_g_rest, n_unique_y_rest = len(set(g_rest)), len(set(y_rest))
    n_splits_2 = min(max(2, round(1 / (1 - val_frac_of_rest))), n_unique_g_rest, n_unique_y_rest)
    if n_splits_2 < 2:
        raise ValueError(f"Grup/kelas tidak cukup untuk split tahap 2 (grup={n_unique_g_rest}, kelas={n_unique_y_rest})")

    sgkf2 = StratifiedGroupKFold(n_splits=n_splits_2, shuffle=True, random_state=random_state)
    X_rest = X[rest_idx]
    val_local, test_local = next(iter(sgkf2.split(X_rest, y_rest, g_rest)))
    val_idx, test_idx = rest_idx[val_local], rest_idx[test_local]

    to_files   = lambda idx: [files[i] for i in idx]
    n_patients = lambda idx: len(set(g[idx]))

    info = {
        'train': {'files': len(train_idx), 'patients': n_patients(train_idx), 'tumor': int(y[train_idx].sum()), 'no_tumor': int((1-y[train_idx]).sum())},
        'val'  : {'files': len(val_idx),   'patients': n_patients(val_idx),   'tumor': int(y[val_idx].sum()),   'no_tumor': int((1-y[val_idx]).sum())},
        'test' : {'files': len(test_idx),  'patients': n_patients(test_idx),  'tumor': int(y[test_idx].sum()),  'no_tumor': int((1-y[test_idx]).sum())},
    }
    return to_files(train_idx), to_files(val_idx), to_files(test_idx), info


## 4. Inventarisasi Dataset


In [ ]:
glioma_imgs     = get_images(glioma)
meningioma_imgs = get_images(meningioma)
scraped_tumor   = glioma_imgs + meningioma_imgs
masked_tumor    = get_images(image_tumor_mask)
no_tumor_imgs   = get_images(no_tumor)

# Pasangkan image_tumor_mask <-> mask_tumor via nomor akhir nama file
mask_map = {}
for img in masked_tumor:
    num = Path(img).stem.rsplit('_', 1)[-1]
    if not num.isdigit():
        continue
    for ext in IMAGE_EXTENSIONS:
        candidate = Path(mask_tumor) / f"mask_{num}{ext}"
        if candidate.exists():
            mask_map[str(img)] = candidate
            break

if len(mask_map) == 0 and len(get_images(mask_tumor)) > 0:
    print("mask_map kosong meskipun folder mask_tumor berisi file. Pastikan sel rename sudah dijalankan.")

total_tumor    = len(scraped_tumor) + len(masked_tumor)
total_no_tumor = len(no_tumor_imgs)
total_all      = total_tumor + total_no_tumor
imbalance      = abs(total_tumor - total_no_tumor) / total_all if total_all else 0

print(f"Total tumor: {total_tumor} | no_tumor: {total_no_tumor} | keseluruhan: {total_all}")
if imbalance > 0.15:
    print(f"Dataset tidak seimbang ({total_tumor} tumor vs {total_no_tumor} no_tumor)")


## 5. Splitting (StratifiedGroupKFold)


In [ ]:
# Gabungkan glioma+meningioma+no_tumor jadi 1 pool agar proporsi kelas
# dijaga secara global (bukan per-sumber) saat StratifiedGroupKFold.
scraped_files, scraped_labels, scraped_groups = [], [], []
for fpath in glioma_imgs:
    scraped_files.append(fpath); scraped_labels.append(1); scraped_groups.append(f"glioma__{get_patient_id(fpath)}")
for fpath in meningioma_imgs:
    scraped_files.append(fpath); scraped_labels.append(1); scraped_groups.append(f"meningioma__{get_patient_id(fpath)}")
for fpath in no_tumor_imgs:
    scraped_files.append(fpath); scraped_labels.append(0); scraped_groups.append(f"no_tumor__{get_patient_id(fpath)}")

sc_train, sc_val, sc_test, sc_info = split_stratified_group(
    scraped_files, scraped_labels, scraped_groups, TRAIN_RATIO, VAL_RATIO, random_state=RANDOM_SEED
)

# image_tumor_mask (1 sumber, tanpa struktur pasien) dikontrol manual via IMAGE_TUMOR_MASK_MODE
masked_shuffled = masked_tumor.copy()
np.random.default_rng(RANDOM_SEED).shuffle(masked_shuffled)

if IMAGE_TUMOR_MASK_MODE == "test_only":
    mt_train, mt_val, mt_test = [], [], list(masked_shuffled)
elif IMAGE_TUMOR_MASK_MODE == "val_test":
    half = len(masked_shuffled) // 2
    mt_train, mt_val, mt_test = [], list(masked_shuffled[:half]), list(masked_shuffled[half:])
elif IMAGE_TUMOR_MASK_MODE == "free":
    n_tr = int(len(masked_shuffled) * TRAIN_RATIO)
    n_v  = int(len(masked_shuffled) * VAL_RATIO)
    mt_train, mt_val, mt_test = list(masked_shuffled[:n_tr]), list(masked_shuffled[n_tr:n_tr+n_v]), list(masked_shuffled[n_tr+n_v:])

file_to_label = {str(f): l for f, l in zip(scraped_files, scraped_labels)}
def partition_split(split_files):
    return ([f for f in split_files if file_to_label.get(str(f), -1) == 1],
            [f for f in split_files if file_to_label.get(str(f), -1) == 0])

sc_train_tumor, sc_train_no = partition_split(sc_train)
sc_val_tumor,   sc_val_no   = partition_split(sc_val)
sc_test_tumor,  sc_test_no  = partition_split(sc_test)

splits = {
    'train': {'tumor': sc_train_tumor + mt_train, 'no_tumor': sc_train_no},
    'val'  : {'tumor': sc_val_tumor   + mt_val,   'no_tumor': sc_val_no},
    'test' : {'tumor': sc_test_tumor  + mt_test,  'no_tumor': sc_test_no},
}

# Verifikasi rasio aktual vs target (SGKF hanya menjamin aproksimasi, bukan rasio persis)
_total_scraped = len(scraped_files)
if _total_scraped > 0:
    _TOLERANCE = 0.07
    _targets  = {'train': TRAIN_RATIO, 'val': VAL_RATIO, 'test': TEST_RATIO}
    _sc_sizes = {'train': len(sc_train), 'val': len(sc_val), 'test': len(sc_test)}
    for _sname, _target in _targets.items():
        _dev = _sc_sizes[_sname] / _total_scraped - _target
        if abs(_dev) > _TOLERANCE:
            print(f"Rasio {_sname} deviasi {_dev*100:+.1f}% dari target, periksa data")


In [ ]:
# Cek tidak ada pasien yang muncul di >1 split (data leakage)
split_patient_map = defaultdict(set)
for sname, cls in splits.items():
    for flist in cls.values():
        for f in flist:
            src_folder = Path(f).parent.name
            if src_folder in ('glioma', 'meningioma', 'no_tumor'):
                split_patient_map[f"{src_folder}__{get_patient_id(f)}"].add(sname)

leaked = {pid: spl for pid, spl in split_patient_map.items() if len(spl) > 1}
if leaked:
    print(f"{len(leaked)} pasien bocor ke lebih dari 1 split!")
    for pid, spl in list(leaked.items())[:5]:
        print(f"   {pid} -> {spl}")

for sname, cls in splits.items():
    n_t, n_nt = len(cls['tumor']), len(cls['no_tumor'])
    n = n_t + n_nt
    if n and abs(n_t / n * 100 - n_nt / n * 100) > 15:
        print(f"{sname.upper()} tidak seimbang: tumor {n_t/n*100:.1f}% vs no_tumor {n_nt/n*100:.1f}%")


## 6. Copy Files ke Output


In [ ]:
from tqdm.notebook import tqdm
import shutil

if COPY_FILES:
    output_path = Path(OUTPUT_ROOT)
    if output_path.exists():
        shutil.rmtree(output_path)  # bersihkan output lama
    output_path.mkdir(parents=True, exist_ok=True)

manifest_rows = []
for split_name, classes in splits.items():
    for cls_name, files in classes.items():
        dest_dir = Path(OUTPUT_ROOT) / split_name / cls_name
        if COPY_FILES:
            for f in tqdm(files, desc=f"{split_name}/{cls_name}", leave=False):
                copy_file_safe(f, dest_dir)
                # Nama file di manifest = nama aktual di disk (folder__stem.png)
                manifest_rows.append({
                    'split': split_name, 'class': cls_name, 'source': Path(f).parent.name,
                    'filename': Path(f).parent.name + '__' + Path(f).stem + OUTPUT_EXT,
                    'has_mask': 'yes' if str(f) in mask_map else 'no',
                })

    # Copy mask pasangan untuk gambar di split ini
    masked_in_split = [f for f in classes['tumor'] if str(f) in mask_map]
    if masked_in_split and COPY_FILES:
        mask_dest = Path(OUTPUT_ROOT) / split_name / 'tumor' / 'masks'
        for img in tqdm(masked_in_split, desc=f"{split_name}/tumor/masks", leave=False):
            copy_file_safe(mask_map[str(img)], mask_dest)

print("Copy selesai." if COPY_FILES else "Dry-run, tidak ada file dicopy.")


## 7. Simpan Manifest & Config Eksperimen


In [ ]:
if COPY_FILES:
    output_path = Path(OUTPUT_ROOT)
    output_path.mkdir(parents=True, exist_ok=True)

    if SAVE_MANIFEST and manifest_rows:
        manifest_path = output_path / 'split_manifest.csv'
        with open(manifest_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=['split', 'class', 'source', 'filename', 'has_mask'])
            writer.writeheader()
            writer.writerows(manifest_rows)

    config_out = {
        'timestamp': datetime.now().isoformat(),
        'IMAGE_TUMOR_MASK_MODE': IMAGE_TUMOR_MASK_MODE,
        'TRAIN_RATIO': TRAIN_RATIO, 'VAL_RATIO': VAL_RATIO, 'TEST_RATIO': TEST_RATIO,
        'RANDOM_SEED': RANDOM_SEED, 'OUTPUT_EXT': OUTPUT_EXT,
        'total_tumor': len(scraped_tumor) + len(masked_tumor),
        'total_no_tumor': len(no_tumor_imgs),
        'split_counts': {sname: {cls: len(files) for cls, files in cls_dict.items()} for sname, cls_dict in splits.items()},
    }
    config_path = output_path / 'split_config.json'
    with open(config_path, 'w') as f:
        json.dump(config_out, f, indent=2)

    print(f"Manifest & config tersimpan di: {output_path}")


## 8. Ringkasan Akhir


In [ ]:
# Ringkasan akhir — satu-satunya laporan lengkap yang dicetak
print("RINGKASAN SPLIT")
print(f"Mode: {IMAGE_TUMOR_MASK_MODE} | Seed: {RANDOM_SEED}")
print(f"{'Split':<8} {'Tumor':>8} {'No Tumor':>10} {'Total':>8} {'Masks':>8}  {'%':>6}")

grand_tumor = grand_no_tumor = 0
for sname, cls in splits.items():
    n_t, n_nt = len(cls['tumor']), len(cls['no_tumor'])
    n_tot = n_t + n_nt
    n_m   = len([f for f in cls['tumor'] if str(f) in mask_map])
    pct   = f"{n_tot/total_all*100:.1f}%" if total_all else "-"
    grand_tumor    += n_t
    grand_no_tumor += n_nt
    print(f"{sname.upper():<8} {n_t:>8} {n_nt:>10} {n_tot:>8} {n_m:>8}  {pct:>6}")

print(f"{'TOTAL':<8} {grand_tumor:>8} {grand_no_tumor:>10} {grand_tumor+grand_no_tumor:>8}")

imbalance = abs(grand_tumor - grand_no_tumor) / total_all if total_all else 0
if imbalance > 0.15:
    print("Dataset tidak seimbang. Pertimbangkan class_weight, augmentasi, atau oversampling.")

print(f"Output: {OUTPUT_ROOT}")
